# PowerOps_v1 — Notebook 05: RAG Question Answering

This notebook wraps Notebook 04's hybrid retrieval with an LLM
answer-generation step, producing `ask_powerops(question)` — the first
end-to-end question-answering function in PowerOps.

```text
Question
   |
Query/metadata parsing        (Notebook 04)
   |
Pinecone retrieval            (Notebook 04)
   |
Relevant DevOps records
   |
Context construction          <- new in this notebook
   |
LLM                           <- new in this notebook
   |
Grounded response
```

### Retrieval recap

The cells below reproduce (condensed, without the walkthrough prints)
`parse_query_filters()` and `retrieve_powerops_documents()` exactly as built
and tested in Notebook 04 — same deterministic vocabulary-based filter
parser, same two-tier assignee disambiguation. See that notebook for the
full explanation of *why* it's deterministic rather than LLM-based.


## 1. Configuration, vocabulary, and Pinecone connection

In [1]:
import json
import os
import re
from pathlib import Path

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone

load_dotenv(dotenv_path=Path("../.env"))

PINECONE_API_KEY = os.environ["PINECONE_API_KEY"]
PINECONE_INDEX_NAME = os.environ.get("PINECONE_INDEX_NAME", "powerops-v1")
EMBEDDING_MODEL = os.environ.get("EMBEDDING_MODEL", "text-embedding-3-small")
CHAT_MODEL = os.environ.get("CHAT_MODEL", "gpt-4o-mini")
LLM_TEMPERATURE = float(os.environ.get("LLM_TEMPERATURE", "0.0"))
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
TOP_K = int(os.environ.get("TOP_K", "5"))

with open("../data/vocabulary.json", "r", encoding="utf-8") as f:
    vocabulary = json.load(f)

pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(PINECONE_INDEX_NAME)
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL, api_key=OPENAI_API_KEY)
vector_store = PineconeVectorStore(index=index, embedding=embeddings)
llm = ChatOpenAI(model=CHAT_MODEL, temperature=LLM_TEMPERATURE)

stats = index.describe_index_stats()
print(f"Connected to '{PINECONE_INDEX_NAME}' — {stats['total_vector_count']} vectors.")
print(f"Chat model: {CHAT_MODEL} (temperature={LLM_TEMPERATURE})")


Connected to 'powerops-v1' — 1000 vectors.
Chat model: gpt-4o-mini (temperature=0.0)


## 2. Retrieval functions (from Notebook 04)

In [2]:
ISSUE_KEY_PATTERN = re.compile(r"\bINO-\d+\b", re.IGNORECASE)


def find_issue_key(question: str) -> str | None:
    match = ISSUE_KEY_PATTERN.search(question)
    return match.group(0).upper() if match else None


def find_vocab_match(question: str, values: list[str]) -> str | None:
    q_lower = question.lower()
    for value in sorted(values, key=len, reverse=True):
        pattern = r"\b" + re.escape(value.lower()) + r"\b"
        if re.search(pattern, q_lower):
            return value
    return None


def _split_name(assignee: str) -> tuple[str, str]:
    cleaned = assignee.replace("(Contractor)", "").strip()
    if "," in cleaned:
        last, first = (p.strip() for p in cleaned.split(",", 1))
    else:
        parts = cleaned.split()
        last, first = (parts[0], " ".join(parts[1:])) if parts else ("", "")
    return last, first


NAME_PARTS = {a: _split_name(a) for a in vocabulary["assignees"]}


def find_assignee(question: str) -> dict:
    q_lower = question.lower()

    full_matches = set()
    for assignee, (last, first) in NAME_PARTS.items():
        if not last or not first:
            continue
        if (re.search(r"\b" + re.escape(last.lower()) + r"\b", q_lower)
                and re.search(r"\b" + re.escape(first.lower()) + r"\b", q_lower)):
            full_matches.add(assignee)

    if len(full_matches) == 1:
        return {"assignee": next(iter(full_matches)), "ambiguous_candidates": None}
    if len(full_matches) > 1:
        return {"assignee": None, "ambiguous_candidates": sorted(full_matches)}

    token_matches = set()
    for assignee, (last, first) in NAME_PARTS.items():
        for token in (first, last):
            if token and len(token) >= 3 and re.search(r"\b" + re.escape(token.lower()) + r"\b", q_lower):
                token_matches.add(assignee)
                break

    if len(token_matches) == 1:
        return {"assignee": next(iter(token_matches)), "ambiguous_candidates": None}
    if len(token_matches) > 1:
        return {"assignee": None, "ambiguous_candidates": sorted(token_matches)}

    return {"assignee": None, "ambiguous_candidates": None}


def parse_query_filters(question: str) -> dict:
    filters = {}
    matched_spans = []

    issue_key = find_issue_key(question)
    if issue_key:
        filters["issue_key"] = issue_key
        matched_spans.append(issue_key)

    team = find_vocab_match(question, vocabulary["assigned_teams"])
    if team:
        filters["assigned_team"] = team
        matched_spans.append(team)

    priority = find_vocab_match(question, vocabulary["priorities"])
    if priority:
        filters["priority"] = priority
        matched_spans.append(priority)

    status = find_vocab_match(question, vocabulary["statuses"])
    if status:
        filters["status"] = status
        matched_spans.append(status)

    assignee_result = find_assignee(question)
    if assignee_result["assignee"]:
        filters["assignee"] = assignee_result["assignee"]
        last, first = NAME_PARTS[assignee_result["assignee"]]
        matched_spans.extend([t for t in (last, first) if t])

    semantic_query = question
    for span in matched_spans:
        semantic_query = re.sub(re.escape(span), "", semantic_query, flags=re.IGNORECASE)
    semantic_query = re.sub(r"\s+", " ", semantic_query).strip(" ?.")
    if not semantic_query:
        semantic_query = question

    return {
        "filters": filters,
        "ambiguous_assignee_candidates": assignee_result["ambiguous_candidates"],
        "semantic_query": semantic_query,
    }


def build_pinecone_filter(filters: dict) -> dict:
    pinecone_filter = {}
    for key in ("assigned_team", "assignee", "priority", "status", "issue_key"):
        if key in filters:
            pinecone_filter[key] = {"$eq": filters[key]}
    return pinecone_filter


def retrieve_powerops_documents(question: str, top_k: int = TOP_K) -> dict:
    parsed = parse_query_filters(question)
    pinecone_filter = build_pinecone_filter(parsed["filters"])

    results = vector_store.similarity_search_with_score(
        parsed["semantic_query"],
        k=top_k,
        filter=pinecone_filter or None,
    )

    return {
        "question": question,
        "filters": parsed["filters"],
        "ambiguous_assignee_candidates": parsed["ambiguous_assignee_candidates"],
        "semantic_query": parsed["semantic_query"],
        "pinecone_filter": pinecone_filter,
        "results": results,
    }


print("Retrieval functions loaded.")


Retrieval functions loaded.


## 3. System prompt

The system prompt is the single most important defense against
hallucination here: it constrains the model to the supplied context, forces
`Issue Key` citations, and gives it explicit permission (and instruction) to
say "not enough evidence" rather than filling gaps.


In [3]:
SYSTEM_PROMPT = """You are PowerOps, a DevOps management assistant.

Answer the user's question using ONLY the supplied PowerOps context below.

Do not invent issues, statuses, assignees, priorities, teams, dates, or \
resolutions that are not explicitly present in the context.

When referencing an issue, include its Issue Key (format: INO-#####).

If the available context does not contain enough evidence to answer the \
question, explicitly state that sufficient information was not found in \
the PowerOps knowledge base. Do not guess or fill gaps.

For questions requesting multiple issues, provide a concise table when \
appropriate (columns: Issue Key, Summary, Team, Assignee, Priority, Status)."""

print(SYSTEM_PROMPT)


You are PowerOps, a DevOps management assistant.

Answer the user's question using ONLY the supplied PowerOps context below.

Do not invent issues, statuses, assignees, priorities, teams, dates, or resolutions that are not explicitly present in the context.

When referencing an issue, include its Issue Key (format: INO-#####).

If the available context does not contain enough evidence to answer the question, explicitly state that sufficient information was not found in the PowerOps knowledge base. Do not guess or fill gaps.

For questions requesting multiple issues, provide a concise table when appropriate (columns: Issue Key, Summary, Team, Assignee, Priority, Status).


## 4. Context construction

Each retrieved Document's `page_content` (built in Notebook 02) is already a
clean, labeled text block — Issue Key, Summary, Team, Assignee, Priority,
Status, dates. We simply join the retrieved documents together, separated
clearly, so the model can tell where one issue ends and the next begins.


In [4]:
def build_context(docs: list) -> str:
    """Render retrieved Documents into a single context block for the LLM."""
    blocks = [f"[Result {i + 1}]\n{doc.page_content}" for i, doc in enumerate(docs)]
    return "\n\n---\n\n".join(blocks)


_sample_docs = [doc for doc, _ in retrieve_powerops_documents(
    "What high priority issues are assigned to Falcon Squad?", top_k=2
)["results"]]
print(build_context(_sample_docs))


[Result 1]
Issue Key: INO-20549
Summary: PCP5 - Please share the  PCP Auto Assignment Request file  from 04/14/2026
Assigned Team: Falcon Squad
Assignee: Mason, Anthony
Reporter: Venkatesh, Angela (Contractor)
Priority: High
Status: Done
Story Points: 0.5
Last Updated: 5/8/26 11:07
Due Date: 4/16/26 0:00

---

[Result 2]
Issue Key: INO-21149
Summary: Please update on listed env. - Non Timeshift
Assigned Team: Falcon Squad
Assignee: Mason, Ashley (Contractor)
Reporter: Webb, Amy
Priority: High
Status: Done
Story Points: Not set
Last Updated: 4/30/26 18:09
Due Date: 4/29/26 0:00


## 5. `ask_powerops()` — the full workflow

One deliberate design choice: if retrieval returns **zero** documents, we
never call the LLM at all. There is nothing for it to ground an answer in,
so skipping straight to "insufficient evidence" is both cheaper and more
reliable than hoping the model says the right thing on its own — this is
the same "don't trust the LLM's self-assessment alone" principle Notebook 06
formalizes for every other evidence check.

The `needs_escalation` flag here is intentionally simple (`True` only when
zero documents were retrieved) — Notebook 06 replaces this with the full
deterministic evidence-evaluation logic (weak matches, requested-but-missing
issue keys, ambiguous assignees, unsupported citations, etc.). This version
exists so `ask_powerops()` has a complete, usable response shape now.


In [5]:
def ask_powerops(question: str, top_k: int = TOP_K) -> dict:
    """Answer a PowerOps question, grounded only in retrieved DevOps context.

    Returns:
        {
            "question": str,
            "answer": str,
            "sources": list[str],       # retrieved Issue Keys
            "filters": dict,            # structured filters detected in the question
            "retrieved_count": int,
            "needs_escalation": bool,
        }
    """
    retrieval = retrieve_powerops_documents(question, top_k=top_k)
    docs = [doc for doc, _score in retrieval["results"]]
    sources = sorted({doc.metadata["issue_key"] for doc in docs})

    if not docs:
        return {
            "question": question,
            "answer": (
                "I could not find enough information in the PowerOps knowledge base "
                "to answer this question reliably. This request should be escalated "
                "to DevOps management."
            ),
            "sources": [],
            "filters": retrieval["filters"],
            "retrieved_count": 0,
            "needs_escalation": True,
        }

    context = build_context(docs)
    messages = [
        ("system", SYSTEM_PROMPT),
        ("human", f"PowerOps Context:\n\n{context}\n\nQuestion: {question}"),
    ]
    response = llm.invoke(messages)

    return {
        "question": question,
        "answer": response.content,
        "sources": sources,
        "filters": retrieval["filters"],
        "retrieved_count": len(docs),
        "needs_escalation": False,
    }


print("ask_powerops() defined.")


ask_powerops() defined.


## 6. Test questions

A mix of questions that should be answerable from this dataset, and
questions that reference things that genuinely don't exist here (a fake
team, a fake issue key, a person not in the data) — the latter should
produce the "insufficient evidence" response, not a hallucinated answer.


In [6]:
ANSWERABLE_QUESTIONS = [
    "What critical issues are currently open?",
    "Summarize Falcon Squad's unresolved issues.",
    "Which high-priority problems belong to Nova Team?",
    "What is happening with INO-21920?",
    "What networking problems are open?",
    "What are the top operational concerns across all three teams?",
]

UNANSWERABLE_QUESTIONS = [
    "What issues does John currently own?",           # person not in this dataset
    "What is happening with OPS-1015?",                # wrong issue-key prefix
    "Which high-priority problems belong to Team Beta?",  # team doesn't exist
    "What is our disaster recovery RTO for Team Alpha?",  # team + concept not in data
]


def print_response(resp: dict) -> None:
    print("=" * 78)
    print(f"Q: {resp['question']}")
    print("-" * 78)
    print(resp["answer"])
    print("-" * 78)
    print(f"Sources: {resp['sources']}")
    print(f"Filters: {resp['filters']}")
    print(f"Retrieved count: {resp['retrieved_count']} | needs_escalation: {resp['needs_escalation']}")
    print()


print("############ ANSWERABLE QUESTIONS ############\n")
for q in ANSWERABLE_QUESTIONS:
    print_response(ask_powerops(q))


############ ANSWERABLE QUESTIONS ############



Q: What critical issues are currently open?
------------------------------------------------------------------------------
Sufficient information was not found in the PowerOps knowledge base to identify any currently open critical issues. All listed issues are marked as "Done."
------------------------------------------------------------------------------
Sources: ['INO-19775', 'INO-19861', 'INO-19863', 'INO-19866', 'INO-19871']
Filters: {}
Retrieved count: 5 | needs_escalation: False



Q: Summarize Falcon Squad's unresolved issues.
------------------------------------------------------------------------------
Sufficient information was not found in the PowerOps knowledge base to identify any unresolved issues for Falcon Squad. All listed issues are marked as "Done."
------------------------------------------------------------------------------
Sources: ['INO-19899', 'INO-20031', 'INO-20755', 'INO-20837', 'INO-21125']
Filters: {'assigned_team': 'Falcon Squad'}
Retrieved count: 5 | needs_escalation: False



Q: Which high-priority problems belong to Nova Team?
------------------------------------------------------------------------------
Here are the high-priority problems that belong to the Nova Team:

| Issue Key  | Summary                                                  | Team       | Assignee           | Priority | Status |
|------------|----------------------------------------------------------|------------|--------------------|----------|--------|
| INO-20547  | PCP5 - System Logs are not logging the issues from EB    | Nova Team  | Myers, Deepa       | High     | Done   |
| INO-19996  | PCP8: Increase the JVM of EB batch chunker GenericBatch  | Nova Team  | Myers, Deepa       | High     | Done   |
| INO-19870  | Donna Trivedi Linux User Access Review Due 04-20-2026   | Nova Team  | Graham, Aditya     | High     | Done   |
| INO-19871  | Donna Rice Linux User Access Review Due 04-20-2026       | Nova Team  | Chawla, Gregory     | High     | Done   |
| INO-19775  | April 2026 LINUX U

Q: What is happening with INO-21920?
------------------------------------------------------------------------------
The issue INO-21920 is related to SDX cases that are being created but are remaining in a "Delayed Processing Pending" status. It is currently assigned to the Falcon Squad and is being worked on by Deepa Sullivan (Contractor). The priority of this issue is Medium, and its status is In Progress.
------------------------------------------------------------------------------
Sources: ['INO-21920']
Filters: {'issue_key': 'INO-21920'}
Retrieved count: 1 | needs_escalation: False



Q: What networking problems are open?
------------------------------------------------------------------------------
Sufficient information was not found in the PowerOps knowledge base to identify any open networking problems. All listed issues are marked as "Done" or "Rejected."
------------------------------------------------------------------------------
Sources: ['INO-20127', 'INO-20618', 'INO-20951', 'INO-21105', 'INO-21636']
Filters: {}
Retrieved count: 5 | needs_escalation: False



Q: What are the top operational concerns across all three teams?
------------------------------------------------------------------------------
Sufficient information was not found in the PowerOps knowledge base to determine the top operational concerns across all three teams.
------------------------------------------------------------------------------
Sources: ['INO-20173', 'INO-20176', 'INO-21319', 'INO-21556', 'INO-21863']
Filters: {}
Retrieved count: 5 | needs_escalation: False



In [7]:
print("############ QUESTIONS WITH NO EVIDENCE IN THIS DATASET ############\n")
for q in UNANSWERABLE_QUESTIONS:
    print_response(ask_powerops(q))


############ QUESTIONS WITH NO EVIDENCE IN THIS DATASET ############



Q: What issues does John currently own?
------------------------------------------------------------------------------
Sufficient information was not found in the PowerOps knowledge base.
------------------------------------------------------------------------------
Sources: ['INO-20754', 'INO-20755', 'INO-20769', 'INO-21728', 'INO-21731']
Filters: {}
Retrieved count: 5 | needs_escalation: False



Q: What is happening with OPS-1015?
------------------------------------------------------------------------------
Sufficient information was not found in the PowerOps knowledge base.
------------------------------------------------------------------------------
Sources: ['INO-19991', 'INO-20247', 'INO-20447', 'INO-21038', 'INO-21869']
Filters: {}
Retrieved count: 5 | needs_escalation: False



Q: Which high-priority problems belong to Team Beta?
------------------------------------------------------------------------------
Sufficient information was not found in the PowerOps knowledge base regarding Team Beta.
------------------------------------------------------------------------------
Sources: ['INO-20130', 'INO-20216', 'INO-20547', 'INO-21071', 'INO-21149']
Filters: {'priority': 'High'}
Retrieved count: 5 | needs_escalation: False



Q: What is our disaster recovery RTO for Team Alpha?
------------------------------------------------------------------------------
Sufficient information was not found in the PowerOps knowledge base to answer the question regarding the disaster recovery RTO for Team Alpha.
------------------------------------------------------------------------------
Sources: ['INO-19773', 'INO-20138', 'INO-20491', 'INO-21196', 'INO-21634']
Filters: {}
Retrieved count: 5 | needs_escalation: False



## Summary & next steps

- Built `ask_powerops(question, top_k)`: parses structured filters, retrieves
  via Pinecone (metadata filter + semantic search), constructs a labeled
  context block from the retrieved Documents, and calls the configured LLM
  with a grounding-focused system prompt.
- Zero-retrieval questions short-circuit to an honest "insufficient evidence"
  response **without calling the LLM** — cheaper and doesn't depend on the
  model correctly self-reporting uncertainty.
- Tested against both answerable questions (real teams/issues in this
  dataset) and deliberately unanswerable ones (a fictional team, a wrong
  issue-key prefix, a person not in the data) — reviewing the notebook
  output is the best way to see whether the model actually declined to
  answer or hallucinated. If it hallucinated on any of the semantic-fallback
  cases (a real risk when metadata filtering finds nothing but semantic
  search still returns *something* to build a context from), that's exactly
  the failure mode Notebook 06 is built to catch systematically.

**Next: Notebook 06 — Confidence and Human Escalation.** We'll replace the
placeholder `needs_escalation` logic with `evaluate_evidence()` and
`should_escalate()` — deterministic checks covering weak/ambiguous matches,
missing requested issue keys, and unsupported citations — plus
`create_escalation()` to persist escalations to `data/escalations.json`.
